# Create Gauss Centre for Supercomputing (GCS) Large-Scale Project Awards

Creates awards for the **in-kind** computing-time allocations of the Gauss Centre for Supercomputing
(GCS), awarded through its twice-yearly **Large-Scale Project** calls on the national supercomputers
of its three centres (HLRS Stuttgart, JSC Juelich, LRZ Garching). Source: the per-call lists at
https://www.gauss-centre.eu/results/large-scale-projects (33 call pages: calls 1-3 and 6-35;
calls 4 and 5 are not linked). Each entry gives title, PI + institution, centre heading (calls >= 9),
sometimes the centre project id(s) and, from call 23 on, the platform and the computing-time period.
No core-hours and no money are published.

**447 awards** (local run 2026-10-01) from 524 call entries.

**Funder:** every Large-Scale allocation is made by GCS's own peer review / steering committee, so
all rows go to **GCS (F4320331625)**, not to the hosting centre (JSC F3307039080 and LRZ
F4320336863 have funder rows for their own, non-GCS allocations; HLRS has none).

**Prerequisites:** run `scripts/local/gauss_centre_to_s3.py` first. It uploads
`s3://openalex-ingest/awards/gauss_centre/gauss_centre_projects.parquet` (1.4 shrink guard).

**`funder_award_id` (2.1.1):** papers cite the centre project id ("pn49yi", "pr74su",
"Project: pn49ha", "hbn26"), so when GCS prints it we ship it lower-cased (the first id when a project
has accounts at two centres; all ids are kept in the description). A project continued in later
calls under the same id is one award listing all its calls. Entries without a printed id (calls
1-18, 29-35 and a few others) get a stable synthetic key `GCS-LSP-C<call>-<title-slug>`.
89 awards carry a GCS-printed project id; 11 citation stubs collapse exactly (pn49su, pn49yi,
pn68nu, pn73fu, pn73xi, pr27yo, pr53yi, pr74sa, pr74su, pr74yo, pr84mi). The id is shipped as printed
(e.g. `GCS-Lamt`, `pr74yo`); the award id hash lower-cases it.

**Amounts / dates:** in-kind, so `amount`/`currency` are NULL; the description lists each call,
its computing-time period, centre(s) and platform. `funding_type = 'grant'` (consistent with the
other w20 compute funders and NWO's computing-time awards). `start_date`/`end_date` = the published
computing-time period (calls >= 23); otherwise only `start_year` (the call year).

**lead_investigator:** the PI (split in Python, runbook 2.4.1; honorifics such as "Prof. Dr.",
"Dr.-Ing." stripped; the first PI when two are named), institution as affiliation (country left
NULL: a few PIs are at non-German institutions).

**Funder details (Path A, F4320* Crossref-registered):** funder_id `4320331625` (Gauss Centre for
Supercomputing, DE; ror 0585fsj26; doi 10.13039/501100022273).

**Priority:** `555` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.gauss_centre_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/gauss_centre/gauss_centre_projects.parquet`;

In [ ]:
%sql
SELECT first_call, COUNT(*) as n FROM openalex.awards.gauss_centre_raw GROUP BY 1 ORDER BY CAST(first_call AS INT);

## Step 1.5: Inspect raw data

In [ ]:
%sql
DESCRIBE openalex.awards.gauss_centre_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.gauss_centre_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320331625 is a Crossref-registered funder, so it MUST resolve to exactly 1 row in
`openalex.funders.funders`. If 0 rows, STOP and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320331625;

In [ ]:
%sql
SELECT assert_true(COUNT(*) = 1, 'F4320331625 must have exactly one row in openalex.funders.funders') AS funder_ok
FROM openalex.funders.funders
WHERE funder_id = 4320331625;

## Step 2: Create GCS Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.gauss_centre_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320331625  -- Gauss Centre for Supercomputing
),

g AS (
    SELECT r.*, TRIM(r.funder_award_id) AS award_id
    FROM openalex.awards.gauss_centre_raw r
    WHERE r.funder_award_id IS NOT NULL AND TRIM(r.funder_award_id) != ''
),

lead AS (
    -- struct field order MUST match openalex_awards_raw: ...orcid, role_start, affiliation.
    SELECT
        award_id,
        CASE WHEN NULLIF(TRIM(lead_family_name), '') IS NOT NULL THEN
        named_struct(
            'given_name', NULLIF(TRIM(lead_given_name), ''),
            'family_name', NULLIF(TRIM(lead_family_name), ''),
            'orcid', CAST(NULL AS STRING),
            'role_start', CAST(NULL AS DATE),
            'affiliation', named_struct(
                'name', NULLIF(TRIM(institution), ''),
                'country', CAST(NULL AS STRING),
                'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
            )
        ) END AS lead_investigator
    FROM g
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.award_id)))) % 9000000000 as id,
    NULLIF(TRIM(g.title), '') as display_name,
    CONCAT_WS(' ',
        'In-kind allocation of supercomputing time (GCS Large-Scale Project; no monetary award).',
        CONCAT(g.allocation_text, '.'),
        CASE WHEN NULLIF(TRIM(g.project_ids_all), '') IS NOT NULL THEN CONCAT('Project id(s): ', TRIM(g.project_ids_all), '.') END
    ) as description,
    f.funder_id,
    g.award_id as funder_award_id,
    CAST(NULL AS DOUBLE) as amount,
    CAST(NULL AS STRING) as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    'grant' as funding_type,
    'GCS Large-Scale Projects' as funder_scheme,
    'gauss_centre_large_scale' as provenance,
    TRY_TO_DATE(g.start_date, 'yyyy-MM-dd') as start_date,
    TRY_TO_DATE(g.end_date, 'yyyy-MM-dd') as end_date,
    TRY_CAST(g.start_year AS INT) as start_year,
    TRY_CAST(g.end_year AS INT) as end_year,
    l.lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CASE WHEN l.lead_investigator IS NOT NULL THEN array(l.lead_investigator) END as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.award_id)))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM g
JOIN lead l ON l.award_id = g.award_id
CROSS JOIN src_funder f;

In [ ]:
%sql
-- Shape check (2.1.1): centre project ids or GCS-LSP synthetic keys; ids unique; no rows lost.
SELECT
    assert_true(COUNT_IF(NOT (funder_award_id RLIKE '^GCS-LSP-C[0-9]{2}-[a-z0-9-]+$' OR funder_award_id RLIKE '^[A-Za-z][A-Za-z0-9_-]{2,30}$')) = 0,
                'unexpected funder_award_id shape') AS shape_ok,
    assert_true(COUNT(*) = COUNT(DISTINCT id), 'duplicate award ids') AS unique_ok,
    assert_true(COUNT(*) = (SELECT COUNT(*) FROM openalex.awards.gauss_centre_raw), 'rows lost in transform') AS complete_ok
FROM openalex.awards.gauss_centre_awards;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'gauss_centre_large_scale' AND priority = 555;

-- Insert into openalex_awards_raw with priority.
-- Priority 555: direct-from-funder ingest of the GCS Large-Scale Project lists (in-kind computing time).
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    555 as priority
FROM openalex.awards.gauss_centre_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards, COUNT(DISTINCT id) as distinct_ids,
       COUNT_IF(funder_award_id NOT LIKE 'GCS-LSP-%') AS with_project_id
FROM openalex.awards.gauss_centre_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, start_date, end_date, start_year,
       lead_investigator.given_name, lead_investigator.family_name, lead_investigator.affiliation.name,
       LEFT(description, 300) AS description_head
FROM openalex.awards.gauss_centre_awards LIMIT 20;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 (lattice-QCD and CFD groups win in most calls).
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.gauss_centre_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.gauss_centre_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.3 / 6.7 coverage (in-kind: pct_amount 0 by design).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    COUNT(start_date) as has_start_date,
    COUNT(start_year) as has_start_year,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_affiliation
FROM openalex.awards.gauss_centre_awards;

In [ ]:
%sql
SELECT start_year, COUNT(*) AS n
FROM openalex.awards.gauss_centre_awards
GROUP BY 1 ORDER BY 1;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'gauss_centre_large_scale'
GROUP BY provenance, priority;